# Experiment 12 — A Chemically-Typed Neuron: Dale's Law + Dopamine-Gated Plasticity

Every neuron in experiments 08-11 was a leaky integrate-and-fire unit that
returned a bare spike (`0`/`1`) and updated weights that could carry any
sign on any synapse, learned freely and independently. The neuron handed
to this notebook has the same LIF dynamics, plus two fields no earlier
notebook's neuron had: `chemical_tag` (which neurotransmitter this neuron
releases) and `release_amount`, surfaced through a `make_signal()` call
that returns `{"spike", "chemical", "amount"}` instead of a bare int.

That's not cosmetic. A real neuron only ever releases one neurotransmitter
at *all* of its output synapses (**Dale's principle**, empirically
confirmed by Eccles, 1954) — a glutamatergic pyramidal cell is excitatory
everywhere it projects; a GABAergic interneuron is inhibitory everywhere it
projects. No experiment in this track has modeled that: 08-11 let every
synapse learn an independent, freely-signed weight, a simplification the
new neuron's `chemical_tag` field makes possible to remove. And `amount`/
`chemical` riding on the signal itself, rather than being baked into a
weight, opens the door to using the exact same field for something else
biological: a **neuromodulator** (dopamine) broadcast as a literal chemical
release after each trial, gating whether locally-tagged synapses actually
update — three-factor learning (Frémaux & Gerstner, 2016 review), instead
of 08-11's flat scalar broadcast.

Two mechanisms this notebook's chemical-tagged neuron makes possible,
tested individually and combined before adopting anything — same
measure-before-you-claim methodology as experiment 11, including its
sharpest lesson: **verify integrated, not just in isolation** (11's async
settling looked harmless in an isolated toy test and then collapsed
Response once wired into the full pipeline). Both steps happen below.

| mechanism | replaces | citation |
|---|---|---|
| Dale's law: fixed E/I identity per neuron, non-negative recurrent weights, sign carried by the presynaptic neuron's `chemical_tag` | free-signed recurrent weights (08-11) | Dale's principle; Eccles 1954; ~80/20 cortical E:I ratio, Markram et al. 2004 |
| Asymmetric dopamine: bigger burst on success than dip on failure | flat ±1 broadcast (08-11) | Schultz 1997 reward-prediction-error (dopamine firing has a floor near zero, so omission dips are shallower than reward bursts) |

Oja's rule and synchronous settling (both verified in experiment 11) are
kept exactly as they were — this notebook isolates the two *new* variables
the chemical-tagged neuron introduces, not a re-litigation of 11's already-
settled questions.

In [1]:
import math
import random

random.seed(0)

## The neuron, exactly as given — unmodified

The only change from experiment 11's `Neuron` is `chemical_tag` /
`release_amount` / `make_signal()`. LIF dynamics (leak, threshold, reset,
refractory) are untouched. Note there's no `noise_std` parameter on
`step()` here (11's had one) — exploration noise gets injected through the
existing `bias` argument instead, at the `Group` level below, so the
neuron class itself needs zero modification.

In [2]:
class Neuron:
    def __init__(
        self,
        n_inputs,
        weights=None,
        threshold=1.0,
        rest=0.0,
        reset=-0.1,
        tau_m=20.0,
        dt=1.0,
        refractory_ms=3.0,
        chemical_tag="glutamate",
        release_amount=1.0
    ):
        self.weights = (
            list(weights)
            if weights is not None
            else [0.0] * n_inputs
        )

        if len(self.weights) != n_inputs:
            raise ValueError("Number of weights must equal n_inputs.")

        self.n_inputs = n_inputs
        self.threshold = threshold
        self.rest = rest
        self.reset = reset
        self.tau_m = tau_m
        self.dt = dt

        # Chemical information carried by this neuron's spikes.
        self.chemical_tag = chemical_tag
        self.release_amount = release_amount

        self.refractory_steps = round(refractory_ms / dt)
        self.refractory_timer = 0

        self.v = rest
        self.last_input = [0.0] * n_inputs
        self.spiked = False
        self.last_release = None

        self.decay = math.exp(-dt / tau_m)

    def make_signal(self, spike):
        """Create the neuron's outgoing electrical/chemical signal."""
        if spike:
            return {
                "spike": 1,
                "chemical": self.chemical_tag,
                "amount": self.release_amount
            }

        return {
            "spike": 0,
            "chemical": None,
            "amount": 0.0
        }

    def step(self, inputs, bias=0.0):
        if len(inputs) != self.n_inputs:
            raise ValueError(
                f"Expected {self.n_inputs} inputs, got {len(inputs)}."
            )

        self.last_input = list(inputs)

        if self.refractory_timer > 0:
            self.refractory_timer -= 1
            self.v = self.reset
            self.spiked = False
            self.last_release = self.make_signal(False)
            return self.last_release

        # Voltage leaks toward resting voltage.
        self.v = self.rest + (self.v - self.rest) * self.decay

        # Integrate incoming signals.
        synaptic_current = sum(
            weight * input_value
            for weight, input_value in zip(self.weights, inputs)
        )

        self.v += synaptic_current + bias

        if self.v >= self.threshold:
            self.v = self.reset
            self.refractory_timer = self.refractory_steps
            self.spiked = True

            self.last_release = self.make_signal(True)
            return self.last_release

        self.spiked = False
        self.last_release = self.make_signal(False)
        return self.last_release

## The exact same 14-turn dataset as experiments 07-11

Moved earlier than experiment 11's layout so the isolated mechanism tests
below can train on the real task instead of a synthetic toy.

In [3]:
train_conversations = [
    [   # A: friendly small talk
        dict(user="hello there friend", intent="greeting", emotion="happy", formality=0.2, closeness=0.6, urgency=0.1,
             tone="playful", plan="answer_directly", response="hello it is good to see you"),
        dict(user="how are you today", intent="question", emotion="happy", formality=0.2, closeness=0.6, urgency=0.1,
             tone="playful", plan="answer_directly", response="i am doing well thank you"),
        dict(user="nice to meet you", intent="greeting", emotion="happy", formality=0.2, closeness=0.6, urgency=0.1,
             tone="playful", plan="answer_directly", response="nice to meet you too"),
        dict(user="okay", intent="statement", emotion="neutral", formality=0.2, closeness=0.6, urgency=0.1,
             tone="playful", plan="answer_directly", response="great let us continue"),
    ],
    [   # B: formal question/answer
        dict(user="what time is the meeting", intent="question", emotion="neutral", formality=0.8, closeness=0.3, urgency=0.3,
             tone="formal", plan="answer_directly", response="the meeting starts at three"),
        dict(user="where is the file", intent="question", emotion="neutral", formality=0.8, closeness=0.3, urgency=0.3,
             tone="formal", plan="ask_clarifying_question", response="which file do you mean"),
    ],
    [   # C: distress -> empathize (the "okay" contrast case lives here)
        dict(user="i am really stressed about this", intent="statement", emotion="anxious", formality=0.3, closeness=0.7, urgency=0.5,
             tone="supportive", plan="empathize", response="that sounds really hard"),
        dict(user="i do not know what to do", intent="statement", emotion="sad", formality=0.3, closeness=0.7, urgency=0.5,
             tone="supportive", plan="empathize", response="i hear you and that matters"),
        dict(user="okay", intent="statement", emotion="neutral", formality=0.2, closeness=0.6, urgency=0.1,
             tone="supportive", plan="empathize", response="take your time i am here for you"),
        dict(user="thank you for listening", intent="statement", emotion="happy", formality=0.3, closeness=0.7, urgency=0.2,
             tone="playful", plan="answer_directly", response="i am glad i could help"),
    ],
    [   # D: commands
        dict(user="please close the door", intent="command", emotion="neutral", formality=0.6, closeness=0.4, urgency=0.4,
             tone="formal", plan="give_instruction", response="closing the door now"),
        dict(user="turn off the lights", intent="command", emotion="neutral", formality=0.6, closeness=0.4, urgency=0.4,
             tone="formal", plan="give_instruction", response="turning off the lights"),
    ],
    [   # E: ambiguous -> clarify -> instruct
        dict(user="can you fix it", intent="question", emotion="neutral", formality=0.4, closeness=0.5, urgency=0.5,
             tone="urgent", plan="ask_clarifying_question", response="which one do you mean"),
        dict(user="the printer upstairs", intent="statement", emotion="neutral", formality=0.4, closeness=0.5, urgency=0.6,
             tone="urgent", plan="give_instruction", response="restart the device now"),
    ],
]

probe_conversation = [
    dict(user="this is not working at all", formality=0.3, closeness=0.4, urgency=0.7),
    dict(user="still broken", formality=0.3, closeness=0.4, urgency=0.8),
]

intents = ["question", "statement", "greeting", "command"]
emotions = ["neutral", "happy", "sad", "angry", "anxious"]
tones = ["supportive", "formal", "playful", "urgent"]
plans = ["answer_directly", "ask_clarifying_question", "empathize", "give_instruction"]
responses = [t["response"] for conv in train_conversations for t in conv]

real_words = sorted({w for conv in train_conversations for t in conv for w in t["user"].split()})
word_to_idx = {w: i for i, w in enumerate(real_words)}
n_turns = sum(len(c) for c in train_conversations)
W = len(real_words)
print(f"{n_turns} turns, {W} distinct words, {len(responses)} unique responses")


def word_pattern(sentence):
    v = [0.0] * W
    for w in sentence.split():
        if w in word_to_idx:
            v[word_to_idx[w]] = 1.0
    return v

14 turns, 41 distinct words, 14 unique responses


## `Group`: Dale's law + dopamine-gated three-factor plasticity

Each neuron in a `Group` is fixed at construction as either glutamatergic
(`"glutamate"`, excitatory) or GABAergic (`"gaba"`, inhibitory) in a fixed
80/20 ratio (Markram et al. 2004's estimate for cortical microcircuits).
That identity, not the sign of a weight, decides whether a neuron's spike
pushes the rest of the group up or down: `_signed()` reads a neuron's own
`chemical_tag` off the very `make_signal()` dict the given `Neuron` class
already produces, and turns a spike into `+amount` (glutamate) or
`-amount` (GABA). Weights themselves become non-negative *magnitudes* —
"how strongly do I listen to this channel" — with `dales_law=True`
clipping recurrent (neuron-to-neuron) weights at zero after every update so
a synapse's sign can never flip. This constraint is scoped to each group's
*own* recurrent synapses only, where the presynaptic side is a real,
identified `Neuron` object — external sensory drive (word/social features)
and `InterGroupLink` charge stay free-signed floats exactly as before, the
same scoping choice experiment 11 made when it isolated Oja's rule to
intra-group updates.

Plasticity keeps 11's tag-and-broadcast skeleton (a local eligibility tag
marks synapses active exactly when their neuron fired) and 11's verified
Oja update shape, but the broadcast itself is now a **dopamine amount**
rather than a bare ±1 scalar: `dopamine = +1.0` on a correct trial,
`dopamine = -1.0` (symmetric) or `-0.3` (asymmetric, `dopamine_asymmetric=
True`) on an incorrect one. The asymmetric option models a real, specific
finding (Schultz, 1997): phasic dopamine neurons fire in a fast, high-
amplitude burst to unexpected reward, but firing rate has a floor near
zero, so the dip below baseline on reward omission is necessarily shallower
than the burst — reward and its absence are not coded symmetrically.
One tagging fix this makes necessary: 08-11 only tagged a synapse when
`last_input[k] > 0` (fine when every input is a non-negative word/social
feature); with Dale's law, recurrent presynaptic input can now be
*negative* (an active inhibitory neuron), so the condition here is
`!= 0` — eligibility should mark "this synapse was causally active,"
excitatory or inhibitory, matching how inhibitory synapses are known to
plasticize too (Vogels et al. 2011, inhibitory STDP).

In [4]:
class Group:
    def __init__(self, n_external, n_neurons, concept_names, noise_std=0.3,
                 dales_law=True, excitatory_frac=0.8, dopamine_asymmetric=True):
        self.n_external = n_external
        self.n_neurons = n_neurons
        self.concept_names = concept_names
        self.noise_std = noise_std
        self.dales_law = dales_law
        self.dopamine_asymmetric = dopamine_asymmetric

        if dales_law:
            n_excitatory = round(n_neurons * excitatory_frac)
            chem_tags = ["glutamate"] * n_excitatory + ["gaba"] * (n_neurons - n_excitatory)
            random.shuffle(chem_tags)
            self.chemical_tags = chem_tags
        else:
            self.chemical_tags = ["glutamate"] * n_neurons

        self.neurons = [
            Neuron(n_inputs=n_external + n_neurons, chemical_tag=self.chemical_tags[i])
            for i in range(n_neurons)
        ]
        self.concept_patterns = {}

    def _reset(self):
        for n in self.neurons:
            n.v = n.rest
            n.refractory_timer = 0

    def _signed(self, neuron, signal):
        if not signal["spike"]:
            return 0.0
        sign = 1.0 if neuron.chemical_tag == "glutamate" else -1.0
        return sign * signal["amount"]

    def _run(self, external, T, explore, bias=None):
        self._reset()
        group_state = [0.0] * self.n_neurons
        spike_counts = [0] * self.n_neurons
        tags = [[False] * len(n.weights) for n in self.neurons]
        bias = bias or [0.0] * self.n_neurons
        for _ in range(T):
            combined = list(external) + group_state
            new_state = []
            for i, neuron in enumerate(self.neurons):
                b = bias[i] + (random.gauss(0, self.noise_std) if explore else 0.0)
                signal = neuron.step(combined, bias=b)
                new_state.append(self._signed(neuron, signal))
                spike_counts[i] += signal["spike"]
                if signal["spike"]:
                    for k in range(len(neuron.weights)):
                        if neuron.last_input[k] != 0:
                            tags[i][k] = True
            group_state = new_state
        return spike_counts, tags

    def _classify_from_counts(self, spike_counts):
        best_c, best_s = None, -1
        for c, pat in self.concept_patterns.items():
            score = sum(a * b for a, b in zip(spike_counts, pat))
            if score > best_s:
                best_s, best_c = score, c
        return best_c

    def train_example(self, external, target_concept, T=15, lr=0.2, bias=None):
        spike_counts, tags = self._run(external, T, explore=True, bias=bias)
        predicted = self._classify_from_counts(spike_counts)
        success = predicted == target_concept
        if self.dopamine_asymmetric:
            dopamine = 1.0 if success else -0.3
        else:
            dopamine = 1.0 if success else -1.0
        for i, neuron in enumerate(self.neurons):
            rate = spike_counts[i] / T
            for k in range(len(neuron.weights)):
                if k == self.n_external + i:
                    continue  # no self-connections
                if tags[i][k]:
                    pre = neuron.last_input[k]
                    new_w = neuron.weights[k] + lr * dopamine * (pre - rate * neuron.weights[k])
                    if self.dales_law and k >= self.n_external:
                        new_w = max(0.0, new_w)  # Dale's law: recurrent synapse strength never sign-flips
                    neuron.weights[k] = new_w
        return predicted, success

    def classify(self, external, bias=None, T=15):
        spike_counts, _ = self._run(external, T, explore=False, bias=bias)
        return self._classify_from_counts(spike_counts), spike_counts

## Test 1: isolated intent-classification sweep

Same methodology experiment 11 used to first screen Oja's rule: an isolated
task (intent classification alone, ignoring the other four groups), 10
seeds, mean accuracy out of 14. Four conditions — free-sign/symmetric (the
08-11-equivalent baseline with this neuron), Dale's law alone, asymmetric
dopamine alone, and both together.

In [5]:
def assign_disjoint_patterns(n_neurons, concept_names, per_concept, seed):
    rng = random.Random(seed)
    order = list(range(n_neurons))
    rng.shuffle(order)
    patterns = {}
    for idx, c in enumerate(concept_names):
        chunk = order[idx * per_concept:(idx + 1) * per_concept]
        pat = [0.0] * n_neurons
        for i in chunk:
            pat[i] = 1.0
        patterns[c] = pat
    return patterns


def sweep_intent(dales_law, dopamine_asymmetric, seeds, n_neurons=16, T=15, lr=0.2, epochs=30):
    flat_turns = [t for conv in train_conversations for t in conv]
    accs = []
    for seed in seeds:
        random.seed(seed)
        g = Group(n_external=W, n_neurons=n_neurons, concept_names=intents, noise_std=0.3,
                  dales_law=dales_law, dopamine_asymmetric=dopamine_asymmetric)
        g.concept_patterns = assign_disjoint_patterns(n_neurons, intents, 4, seed=100)
        for epoch in range(epochs):
            g.noise_std = 0.3 + (0.05 - 0.3) * (epoch / max(1, epochs - 1))
            for t in flat_turns:
                g.train_example(word_pattern(t["user"]), t["intent"], lr=lr)
        g.noise_std = 0.0
        correct = sum(g.classify(word_pattern(t["user"]))[0] == t["intent"] for t in flat_turns)
        accs.append(correct)
    return accs


def mean(xs):
    return sum(xs) / len(xs)


seeds = list(range(10))
sweep_results = {
    "baseline": dict(label="free-sign, symmetric dopamine (08-11 baseline)",
                      accs=sweep_intent(dales_law=False, dopamine_asymmetric=False, seeds=seeds)),
    "dale_only": dict(label="Dale's law only",
                       accs=sweep_intent(dales_law=True, dopamine_asymmetric=False, seeds=seeds)),
    "asym_only": dict(label="asymmetric dopamine only",
                       accs=sweep_intent(dales_law=False, dopamine_asymmetric=True, seeds=seeds)),
    "both": dict(label="Dale's law + asymmetric dopamine",
                 accs=sweep_intent(dales_law=True, dopamine_asymmetric=True, seeds=seeds)),
}
for key, d in sweep_results.items():
    d["mean"] = mean(d["accs"])
    label, accs, m = d["label"], d["accs"], d["mean"]
    print(f"{label:48} {accs}  mean={m:.2f}/14")

isolated_best = max(sweep_results, key=lambda k: sweep_results[k]["mean"])
print(f"\nisolated-test best: {sweep_results[isolated_best]['label']}")

free-sign, symmetric dopamine (08-11 baseline)   [13, 9, 11, 11, 9, 13, 12, 10, 11, 11]  mean=11.00/14
Dale's law only                                  [8, 13, 10, 9, 13, 12, 12, 9, 13, 12]  mean=11.10/14
asymmetric dopamine only                         [9, 10, 6, 5, 10, 9, 8, 8, 8, 6]  mean=7.90/14
Dale's law + asymmetric dopamine                 [6, 5, 6, 9, 8, 7, 8, 7, 8, 6]  mean=7.00/14

isolated-test best: Dale's law only


## Test 2: does the isolated winner hold up on the full pipeline?

Experiment 11's single sharpest lesson: a mechanism that looks safe (or
even good) in an isolated toy test can still fail once wired into the real
five-group pipeline with inter-group charge — that's exactly what happened
to async settling there. Asymmetric dopamine lost decisively in both
isolated conditions above (a large, consistent 10-seed gap, not a
close call), so it isn't re-tested at full-pipeline scale here. Dale's
law's isolated margin, by contrast, is narrow enough (within the seed-to-
seed noise band) that it's exactly the kind of result that needs a real
integrated check before shipping — so that's what this trains: the full
five-group agent, twice, once with Dale's law on and once off, dopamine
symmetric in both (since that half of the sweep was unambiguous), same
seed, same 120-epoch schedule, then compared on all five real tasks.

In [6]:
class InterGroupLink:
    def __init__(self, source, target, lr=0.02, w_max=1.0):
        self.source, self.target = source, target
        self.lr, self.w_max = lr, w_max
        self.weights = [[0.0] * source.n_neurons for _ in range(target.n_neurons)]

    def train(self, source_concept, target_concept):
        s_pat = self.source.concept_patterns[source_concept]
        t_pat = self.target.concept_patterns[target_concept]
        for i in range(self.target.n_neurons):
            if t_pat[i] > 0:
                for j in range(self.source.n_neurons):
                    self.weights[i][j] = min(self.weights[i][j] + self.lr * s_pat[j], self.w_max)

    def charge(self, source_spike_counts):
        return [sum(w * s for w, s in zip(self.weights[i], source_spike_counts))
                for i in range(self.target.n_neurons)]


def assign_similarity_patterns(n_neurons, concept_coords, per_concept, seed):
    rng = random.Random(seed)
    neuron_prefs = [(rng.uniform(-1, 1), rng.uniform(-1, 1)) for _ in range(n_neurons)]
    patterns = {}
    for c, (cx, cy) in concept_coords.items():
        dists = sorted(range(n_neurons), key=lambda i: (neuron_prefs[i][0] - cx) ** 2 + (neuron_prefs[i][1] - cy) ** 2)
        pat = [0.0] * n_neurons
        for i in dists[:per_concept]:
            pat[i] = 1.0
        patterns[c] = pat
    return patterns


emotion_coords = {  # (valence, arousal) -- Russell's circumplex model
    "neutral": (0.0, 0.0), "happy": (0.8, 0.4), "sad": (-0.7, -0.6),
    "angry": (-0.6, 0.8), "anxious": (-0.5, 0.7),
}

MEMORY_DECAY = 0.5
LR = 0.20
N_EPOCHS = 120


class ConversationalAgent:
    def __init__(self, seed=0, noise_std=0.3, dales_law=True, dopamine_asymmetric=False):
        random.seed(seed)
        kw = dict(dales_law=dales_law, dopamine_asymmetric=dopamine_asymmetric)
        self.intent_g = Group(n_external=W, n_neurons=16, concept_names=intents, noise_std=noise_std, **kw)
        self.intent_g.concept_patterns = assign_disjoint_patterns(16, intents, 4, seed=100)
        self.emotion_g = Group(n_external=W, n_neurons=15, concept_names=emotions, noise_std=noise_std, **kw)
        self.emotion_g.concept_patterns = assign_similarity_patterns(15, emotion_coords, per_concept=3, seed=3)

        fused_dim = W * 2 + 3
        self.tone_g = Group(n_external=fused_dim, n_neurons=16, concept_names=tones, noise_std=noise_std, **kw)
        self.tone_g.concept_patterns = assign_disjoint_patterns(16, tones, 4, seed=102)
        self.plan_g = Group(n_external=fused_dim, n_neurons=16, concept_names=plans, noise_std=noise_std, **kw)
        self.plan_g.concept_patterns = assign_disjoint_patterns(16, plans, 4, seed=103)
        self.response_g = Group(n_external=fused_dim, n_neurons=42, concept_names=list(range(len(responses))), noise_std=noise_std, **kw)
        self.response_g.concept_patterns = assign_disjoint_patterns(42, list(range(len(responses))), 3, seed=104)
        self.memory_dim = W

        self.link_emotion_tone = InterGroupLink(self.emotion_g, self.tone_g)
        self.link_emotion_plan = InterGroupLink(self.emotion_g, self.plan_g)
        self.link_emotion_response = InterGroupLink(self.emotion_g, self.response_g)
        self.link_tone_response = InterGroupLink(self.tone_g, self.response_g)
        self.link_plan_response = InterGroupLink(self.plan_g, self.response_g)

    def _fused(self, meaning, memory, social):
        return meaning + memory + list(social)

    def all_groups(self):
        return [self.intent_g, self.emotion_g, self.tone_g, self.plan_g, self.response_g]

    def phase1_train(self, turn, memory):
        meaning = word_pattern(turn["user"])
        social = [turn["formality"], turn["closeness"], turn["urgency"]]
        fused = self._fused(meaning, memory, social)
        self.intent_g.train_example(meaning, turn["intent"], lr=LR)
        self.emotion_g.train_example(meaning, turn["emotion"], lr=LR)
        self.tone_g.train_example(fused, turn["tone"], lr=LR)
        self.plan_g.train_example(fused, turn["plan"], lr=LR)
        self.response_g.train_example(fused, responses.index(turn["response"]), lr=LR)
        return [m * MEMORY_DECAY + x for m, x in zip(memory, meaning)]

    def phase2_train(self, turn):
        response_idx = responses.index(turn["response"])
        self.link_emotion_tone.train(turn["emotion"], turn["tone"])
        self.link_emotion_plan.train(turn["emotion"], turn["plan"])
        self.link_emotion_response.train(turn["emotion"], response_idx)
        self.link_tone_response.train(turn["tone"], response_idx)
        self.link_plan_response.train(turn["plan"], response_idx)

    def step(self, turn, memory):
        meaning = word_pattern(turn["user"])
        social = [turn["formality"], turn["closeness"], turn["urgency"]]
        fused = self._fused(meaning, memory, social)
        predicted_intent, _ = self.intent_g.classify(meaning)
        predicted_emotion, emotion_counts = self.emotion_g.classify(meaning)
        tone_charge = self.link_emotion_tone.charge(emotion_counts)
        plan_charge = self.link_emotion_plan.charge(emotion_counts)
        predicted_tone, tone_counts = self.tone_g.classify(fused, bias=tone_charge)
        predicted_plan, plan_counts = self.plan_g.classify(fused, bias=plan_charge)
        response_charge = [e + t + p for e, t, p in zip(
            self.link_emotion_response.charge(emotion_counts),
            self.link_tone_response.charge(tone_counts),
            self.link_plan_response.charge(plan_counts))]
        response_idx, _ = self.response_g.classify(fused, bias=response_charge)
        new_memory = [m * MEMORY_DECAY + x for m, x in zip(memory, meaning)]
        return new_memory, dict(predicted_intent=predicted_intent, predicted_emotion=predicted_emotion,
                                 predicted_tone=predicted_tone, predicted_plan=predicted_plan,
                                 predicted_response=responses[response_idx])


def anneal_noise(agent, epoch, n_epochs, noise_start=0.3, noise_end=0.05):
    v = noise_start + (noise_end - noise_start) * (epoch / max(1, n_epochs - 1))
    for g in agent.all_groups():
        g.noise_std = v


def build_and_train_agent(dales_law, dopamine_asymmetric, seed=0, n_epochs=N_EPOCHS):
    agent = ConversationalAgent(seed=seed, dales_law=dales_law, dopamine_asymmetric=dopamine_asymmetric)
    for epoch in range(n_epochs):
        anneal_noise(agent, epoch, n_epochs)
        for conv in train_conversations:
            memory = [0.0] * agent.memory_dim
            for turn in conv:
                memory = agent.phase1_train(turn, memory)
    for conv in train_conversations:
        for turn in conv:
            agent.phase2_train(turn)
    for g in agent.all_groups():
        g.noise_std = 0.0
    return agent


def evaluate(agent):
    eval_log = []
    for conv in train_conversations:
        memory = [0.0] * agent.memory_dim
        for turn in conv:
            memory, result = agent.step(turn, memory)
            eval_log.append((turn, result))
    correct = dict(intent=0, emotion=0, tone=0, plan=0, response=0)
    for turn, result in eval_log:
        correct["intent"] += result["predicted_intent"] == turn["intent"]
        correct["emotion"] += result["predicted_emotion"] == turn["emotion"]
        correct["tone"] += result["predicted_tone"] == turn["tone"]
        correct["plan"] += result["predicted_plan"] == turn["plan"]
        correct["response"] += result["predicted_response"] == turn["response"]
    return correct, eval_log


agent_dale_on = build_and_train_agent(dales_law=True, dopamine_asymmetric=False, seed=0)
agent_dale_off = build_and_train_agent(dales_law=False, dopamine_asymmetric=False, seed=0)
correct_on, eval_log_on = evaluate(agent_dale_on)
correct_off, eval_log_off = evaluate(agent_dale_off)

print("full-pipeline, Dale's law ON :", {k: f"{v}/{n_turns}" for k, v in correct_on.items()}, " total", sum(correct_on.values()), "/", 5 * n_turns)
print("full-pipeline, Dale's law OFF:", {k: f"{v}/{n_turns}" for k, v in correct_off.items()}, " total", sum(correct_off.values()), "/", 5 * n_turns)

FINAL_DALES_LAW = sum(correct_on.values()) >= sum(correct_off.values())
FINAL_DOPAMINE_ASYM = False  # ruled out decisively by Test 1, not re-tested at full-pipeline scale
print(f"\nship Dale's law: {FINAL_DALES_LAW}   ship asymmetric dopamine: {FINAL_DOPAMINE_ASYM}")

agent = agent_dale_on if FINAL_DALES_LAW else agent_dale_off
correct, eval_log = (correct_on, eval_log_on) if FINAL_DALES_LAW else (correct_off, eval_log_off)

full-pipeline, Dale's law ON : {'intent': '14/14', 'emotion': '13/14', 'tone': '5/14', 'plan': '8/14', 'response': '1/14'}  total 41 / 70
full-pipeline, Dale's law OFF: {'intent': '14/14', 'emotion': '14/14', 'tone': '11/14', 'plan': '8/14', 'response': '5/14'}  total 52 / 70

ship Dale's law: False   ship asymmetric dopamine: False


## Verdict

**Test 1 (isolated intent classification, 10 seeds):** Dale's law's win
over the free-sign baseline is 11.10 vs 11.00 mean out of 14 -- a tenth of
a point, well inside the 5-13 range individual seeds swing across either
condition. Asymmetric dopamine loses decisively and consistently on its
own (7.90/14) and combined with Dale's law (7.00/14) -- that part of the
verdict is unambiguous and not worth a further integrated check.

**Test 2 (the real check -- full five-group pipeline, same seed, Dale's
law on vs off):**

| task | Dale's law ON | Dale's law OFF |
|---|---|---|
| intent | 14/14 | 14/14 |
| emotion | 13/14 | 14/14 |
| tone | **5/14** | **11/14** |
| plan | 8/14 | 8/14 |
| response | **1/14** | **5/14** |
| **total** | **41/70** | **52/70** |

This is experiment 11's async-settling lesson again, almost exactly: a
mechanism whose isolated-test margin was inside the noise band turned into
a real, substantial regression once wired into the full pipeline -- tone
lost more than half its accuracy and response nearly collapsed entirely
under Dale's law. The likely mechanism: forcing every group's recurrent
weights to be non-negative removes free-sign weights' ability to actively
*suppress* a wrong-direction recurrent trajectory once one takes hold --
with only decay-toward-zero available (never a sign flip), larger,
higher-dimensional groups (tone/plan/response, all driven by an 85-wide
fused input plus three-source inter-group charge) had less room to correct
course than intent/emotion's much simpler, meaning-only 41-wide input.

**Verdict: REVERT Dale's law. Reject asymmetric dopamine (already
decisive from Test 1).** Both `FINAL_DALES_LAW` and `FINAL_DOPAMINE_ASYM`
came back `False`, computed from the numbers above, not assumed. The
shipped configuration is mechanically the same as experiments 08-11 (free-
signed weights, flat +/-1 dopamine, Oja's rule, synchronous settling) --
this notebook's two new candidate mechanisms, motivated by real citations
and implemented correctly, simply didn't earn their place once tested at
the scale that matters.

## Final comparison against experiments 07-11

`agent` below is whichever of the two full-pipeline runs above actually won
— not assumed, read straight out of `FINAL_DALES_LAW`.

In [7]:
print("exp12 (this notebook)  vs  exp11 (researched fixes)  vs  exp10  vs  exp09  vs  exp08  vs  exp07 (backprop):")
exp11 = dict(intent=13/14, emotion=12/14, tone=11/14, plan=9/14, response=6/14)
exp10 = dict(intent=11/14, emotion=12/14, tone=11/14, plan=9/14, response=5/14)
exp09 = dict(intent=1.00, emotion=1.00, tone=12/14, plan=12/14, response=12/14)
exp08 = dict(intent=13/14, emotion=1.00, tone=11/14, plan=10/14, response=13/14)
exp07 = dict(intent=1.00, emotion=1.00, tone=1.00, plan=1.00, response=1.00)
for k, v in correct.items():
    print(f"  {k:9} {v}/{n_turns} = {v/n_turns:.0%}   "
          f"(exp11: {exp11[k]:.0%}, exp10: {exp10[k]:.0%}, exp09: {exp09[k]:.0%}, exp08: {exp08[k]:.0%}, exp07: {exp07[k]:.0%})")

exp12 (this notebook)  vs  exp11 (researched fixes)  vs  exp10  vs  exp09  vs  exp08  vs  exp07 (backprop):
  intent    14/14 = 100%   (exp11: 93%, exp10: 79%, exp09: 100%, exp08: 93%, exp07: 100%)
  emotion   14/14 = 100%   (exp11: 86%, exp10: 86%, exp09: 100%, exp08: 100%, exp07: 100%)
  tone      11/14 = 79%   (exp11: 79%, exp10: 79%, exp09: 86%, exp08: 79%, exp07: 100%)
  plan      8/14 = 57%   (exp11: 64%, exp10: 64%, exp09: 86%, exp08: 71%, exp07: 100%)
  response  5/14 = 36%   (exp11: 43%, exp10: 36%, exp09: 86%, exp08: 93%, exp07: 100%)


In [8]:
print("per-turn breakdown:\n")
for turn, result in eval_log:
    flags = []
    for k in ["intent", "emotion", "tone", "plan"]:
        if result["predicted_" + k] != turn[k]:
            flags.append(f"{k}: pred={result['predicted_' + k]} true={turn[k]}")
    if result["predicted_response"] != turn["response"]:
        flags.append(f"response: pred={result['predicted_response']!r} true={turn['response']!r}")
    print(f"{turn['user']!r:35} {'ALL OK' if not flags else ' | '.join(flags)}")

per-turn breakdown:

'hello there friend'                ALL OK
'how are you today'                 response: pred='i am glad i could help' true='i am doing well thank you'
'nice to meet you'                  response: pred='which file do you mean' true='nice to meet you too'
'okay'                              response: pred='nice to meet you too' true='great let us continue'
'what time is the meeting'          ALL OK
'where is the file'                 plan: pred=answer_directly true=ask_clarifying_question | response: pred='that sounds really hard' true='which file do you mean'
'i am really stressed about this'   ALL OK
'i do not know what to do'          ALL OK
'okay'                              tone: pred=playful true=supportive | plan: pred=answer_directly true=empathize | response: pred='great let us continue' true='take your time i am here for you'
'thank you for listening'           ALL OK
'please close the door'             plan: pred=answer_directly true=give_instruction | 

## Does memory still matter? (the same "okay" test as 07-11)

In [9]:
memory_walk = [0.0] * agent.memory_dim
for turn in train_conversations[2][:2]:
    meaning = word_pattern(turn["user"])
    memory_walk = [m * MEMORY_DECAY + x for m, x in zip(memory_walk, meaning)]
memory_after_t2 = memory_walk

okay_turn = train_conversations[2][2]
_, result_carried = agent.step(okay_turn, memory_after_t2)
_, result_reset = agent.step(okay_turn, [0.0] * agent.memory_dim)

print(f"turn: {okay_turn['user']!r}  (true target response: {okay_turn['response']!r})\n")
print("memory carried (real distress context):")
print(f"  tone={result_carried['predicted_tone']:10} plan={result_carried['predicted_plan']:24} response={result_carried['predicted_response']!r}")
print("memory reset (as if the prior turns never happened):")
print(f"  tone={result_reset['predicted_tone']:10} plan={result_reset['predicted_plan']:24} response={result_reset['predicted_response']!r}")

turn: 'okay'  (true target response: 'take your time i am here for you')

memory carried (real distress context):
  tone=playful    plan=answer_directly          response='great let us continue'
memory reset (as if the prior turns never happened):
  tone=playful    plan=answer_directly          response='the meeting starts at three'


## A conversation it never saw

The exact same held-out probe as experiments 07-11.

In [10]:
memory = [0.0] * agent.memory_dim
for turn in probe_conversation:
    memory, result = agent.step(turn, memory)
    print(f"{turn['user']!r:30} -> emotion={result['predicted_emotion']:8} tone={result['predicted_tone']:10} "
          f"plan={result['predicted_plan']:24} response={result['predicted_response']!r}")

'this is not working at all'   -> emotion=neutral  tone=playful    plan=answer_directly          response='the meeting starts at three'
'still broken'                 -> emotion=neutral  tone=playful    plan=ask_clarifying_question  response='which file do you mean'


## What actually happened

**Neither new mechanism survived contact with the full pipeline.** Dale's
law looked like a plausible, even slightly favorable change in isolation
and then cost 11 percentage points of aggregate accuracy once integrated
(52/70 -> 41/70), concentrated almost entirely in tone and response -- the
two groups most dependent on inter-group charge and highest-dimensional
external input, exactly where a real biological constraint (a synapse's
strength can never reverse the sign fixed by what its neuron releases)
bites hardest against a network that has no other way to unlearn a bad
trajectory. Asymmetric dopamine never had a case to begin with: shifting
the failure signal from -1.0 to -0.3 weakens the error-correcting half of
every single update, and lost by a wide, consistent margin in isolation
before any integration test was needed.

**The shipped agent is therefore mechanically identical to experiment
11's** (Oja's rule, synchronous settling, flat +/-1 dopamine, free-signed
weights) with a fresh training run: intent 100% (14/14), emotion 100%
(14/14), tone 79% (11/14), plan 57% (8/14), response 36% (5/14) -- matches
or beats 11 on intent/emotion/tone, trails a little on plan and response,
within the seed-to-seed noise band established back in experiment 10's
variance sweep (9-11/13 on intent alone), not a regression attributable to
anything changed here. 7 of 14 turns landed exactly right end to end; the
remaining errors are concentrated in the 14-way Response classifier, the
same structural hot spot flagged in every notebook since experiment 10.

**Memory ablation, honestly reported, is weaker this run than 11's:** the
"okay" turn now gets identical tone=playful, plan=answer_directly under
both memory-carried and memory-reset conditions -- only the recalled
response differs ("great let us continue" vs "the meeting starts at
three", neither the true target "take your time i am here for you").
Memory still measurably changes what the agent recalls, but the effect is
narrower than 11's (which saw tone itself flip between conditions) --
expected given plan/response sit at 57%/36%, and consistent with this
being one more data point in the same noisy-reward-modulated-learning
story this whole Hebbian-family track has told since experiment 10, not a
new finding.

**The novel probe broke the four-notebook streak of collapsing to one
generic fallback:** experiments 08-11 all converged both held-out lines to
the *same* recalled response; here the two lines diverge ("the meeting
starts at three" vs "which file do you mean") -- still wrong, still
associative recall rather than composition, but a different failure shape
worth noting rather than a repeat of the earlier pattern.

**Where this leaves things:** the chemically-tagged neuron's `chemical`/
`amount` fields are real and reusable -- they just don't pay for themselves
under a global, uniform Dale's-law constraint applied to every recurrent
synapse in every group. A narrower version (e.g., inhibition scoped only
to the response layer as lateral competition, rather than network-wide)
is a real candidate for a future experiment, in the same spirit as 11
deferring equilibrium propagation / predictive coding / surrogate-gradient
spiking networks to whenever this track next tackles the still-unsolved
"recall, not generation" ceiling shared by every Hebbian-family agent
(08/09/10/11/12) built so far.